# QKD Lab: run the simulation in Google Colab

1. Select **Runtime → Run all** once to load the engine and run the defaults.
2. Change settings in step 2 and press **that same cell's Run button**. Its results and matching links update together.
3. Open the matching dashboard link beneath its output. Both platforms calculate from the same inputs and seed.

This notebook executes an inspectable snapshot of the application's JavaScript engine in a separate Node.js process. It demonstrates reproducibility and portability, not independent verification by another implementation. Final bits are modeled key-length estimates, not extracted secure encryption keys.


## 1. Load the simulation code
The JavaScript source is embedded below. Expand this cell to inspect the actual BB84/B92 and intercept-resend implementation. Node.js executes it in the Colab runtime, outside the website. No quantum hardware or GPU is required.

In [1]:
import json, pathlib, shutil, subprocess, hashlib, sys, math

node = shutil.which("node") or shutil.which("nodejs")
if node is None:
    try:
        import google.colab
    except ImportError:
        raise RuntimeError("Install Node.js 18 or newer, then rerun this notebook.")
    # Only the disposable Colab runtime is changed if Node.js is missing.
    subprocess.run(["apt-get", "update", "-qq"], check=True, capture_output=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "nodejs"], check=True, capture_output=True)
    node = shutil.which("node") or shutil.which("nodejs")
version = subprocess.check_output([node, "--version"], text=True).strip()
if int(version.lstrip("v").split(".")[0]) < 18:
    raise RuntimeError("Node.js 18 or newer is required.")

ENGINE_SOURCE = r"""// Educational prepare-and-measure model; no generated bits are cryptographic keys.
const DEFAULTS = {protocol:'BB84',pulses:10000,noise:.05,attack:'none',interception:1,lossDb:0,efficiency:1,darkCount:0,source:'ideal',mu:.5,sampleFraction:.25,seed:42,eps:1e-6,requiredBits:128};
export function normalize(input={}) {
  const c={...DEFAULTS,...input};
  for (const [key,values] of Object.entries({protocol:['BB84','B92'],attack:['none','ir','pns'],source:['ideal','wcp']}))
    if(!values.includes(c[key])) throw new RangeError(`Invalid ${key}`);
  for(const key of ['noise','interception','efficiency','darkCount','sampleFraction'])
    if(!Number.isFinite(c[key])||c[key]<0||c[key]>1) throw new RangeError(`${key} must be in [0,1]`);
  for(const key of ['pulses','seed','requiredBits']) if(!Number.isSafeInteger(c[key])) throw new RangeError(`${key} must be an integer`);
  if(c.pulses<1||c.pulses>1e6||c.requiredBits<0) throw new RangeError('pulses must be 1..1000000 and requiredBits nonnegative');
  if(!Number.isFinite(c.lossDb)||c.lossDb<0||c.lossDb>100) throw new RangeError('lossDb must be in [0,100]');
  if(!Number.isFinite(c.mu)||c.mu<=0||c.mu>5) throw new RangeError('mu must be in (0,5]');
  if(!Number.isFinite(c.eps)||c.eps<=0||c.eps>=1) throw new RangeError('eps must be in (0,1)');
  if(c.attack==='pns'&&c.source!=='wcp') throw new RangeError('PNS requires a weak coherent pulse source');
  return c;
}
function rng(seed,salt) {
  let state=(seed^salt)>>>0;
  return ()=>{state=(state+0x6D2B79F5)>>>0;let x=state;x=Math.imul(x^(x>>>15),x|1);x^=x+Math.imul(x^(x>>>7),x|61);return ((x^(x>>>14))>>>0)/4294967296;};
}
export function h2(q) {if(q<=0||q>=1)return 0;return -q*Math.log2(q)-(1-q)*Math.log2(1-q);}
export function qberUpper(q,n,eps=1e-6) {
  if(n<=0||q>=.5)return .5;
  const target=Math.log(1/eps)/n;
  let low=q,high=.5;
  // Invert binary relative entropy in natural units (Chernoff/KL bound).
  for(let i=0;i<60;i++){const mid=(low+high)/2;const divergence=q===0?-Math.log1p(-mid):q*Math.log(q/mid)+(1-q)*Math.log((1-q)/(1-mid));if(divergence>target)high=mid;else low=mid;}
  return high;
}
export function gllpRate(q,delta=1,fec=1.16) {return delta<=0||q/delta>=.5?0:Math.max(0,delta*(1-h2(q/delta))-fec*h2(q));}

export function binomialInterval(successes, trials, eps=1e-6) {
  if(!Number.isSafeInteger(trials)||trials<1||!Number.isSafeInteger(successes)||successes<0||successes>trials||!(eps>0&&eps<1)) throw new RangeError('Invalid confidence interval');
  const q=successes/trials,budget=Math.log(2/eps)/trials;
  const kl=p=>(q===0?0:q*Math.log(q/p))+(q===1?0:(1-q)*Math.log((1-q)/(1-p)));
  let lo=0,hi=q;
  for(let i=0;i<60;i++){const mid=(lo+hi)/2;if(kl(mid)>budget)lo=mid;else hi=mid;}
  const lower=q===0?0:lo;lo=q;hi=1;
  for(let i=0;i<60;i++){const mid=(lo+hi)/2;if(kl(mid)>budget)hi=mid;else lo=mid;}
  return [lower,q===1?1:hi];
}

// Source-replacement dual witness for the standard B92 filter (gamma=beta).
// Each witness is an operator upper bound, so incomplete minimization only
// reduces yield. IID confidence intervals are NOT a coherent-attack proof.
export function b92PhaseBound({source='ideal',pulses,detected,sifted,errors,sample,eps=1e-6}) {
  if(source!=='ideal')return {supported:false,phaseUpper:.5,reason:'B92 weak-coherent-source security requires additional tagging/decoy analysis.'};
  if(!sample||!sifted)return {supported:false,phaseUpper:.5,reason:'Insufficient B92 observations.'};
  const a=(1-1/Math.sqrt(2))/2,b=1-a,ab=Math.sqrt(a*b);
  const conclusive=binomialInterval(sifted,pulses,eps/3),loss=binomialInterval(pulses-detected,pulses,eps/3);
  const qInterval=binomialInterval(errors,sample,eps/3),errorUpper=qInterval[1]*conclusive[1];
  const eigen=(u,v,z)=>(u+v)/2+Math.hypot((u-v)/2,z);
  function witness(coefficients) {
    const [le,lc,ll,la]=coefficients,z=le*ab/2;
    const identity=Math.max(
      eigen(-lc*a-le*a/2,-lc*b-la-le*b/2,z),
      eigen(b-lc*b-le*b/2,a-lc*a-la-le*a/2,z),-ll,-ll-la
    )+1e-11*(1+coefficients.reduce((sum,x)=>sum+Math.abs(x),0));
    return {joint:identity+le*errorUpper+lc*conclusive[lc>=0?1:0]+ll*loss[ll>=0?1:0]+la*a,identity};
  }
  let best={joint:Infinity};
  // Bounded deterministic convex coordinate search; no optimum is asserted.
  for(const initial of [[4,0,1,0],[16,-2,4,-1],[64,-8,16,0]]) {
    let coefficients=[...initial];
    for(let round=0;round<16;round++)for(let axis=0;axis<4;axis++) {
      let lo=axis===0?0:-256,hi=256;
      for(let step=0;step<38;step++) {
        const x=lo+(hi-lo)/3,y=hi-(hi-lo)/3;
        const cx=[...coefficients],cy=[...coefficients];cx[axis]=x;cy[axis]=y;
        if(witness(cx).joint<witness(cy).joint)hi=y;else lo=x;
      }
      const next=[...coefficients];next[axis]=(lo+hi)/2;
      if(witness(next).joint<witness(coefficients).joint)coefficients=next;
    }
    const candidate=witness(coefficients);
    if(candidate.joint<best.joint)best={...candidate,coefficients};
  }
  // Refine coupled coefficients: coordinate search alone can stall on the
  // nondifferentiable eigenvalue envelope. Every returned point is still a witness.
  const score=c=>c[0]<0||c.some(x=>!Number.isFinite(x)||Math.abs(x)>1e6)?Infinity:witness(c).joint;
  let simplex=[best.coefficients,...best.coefficients.map((_,axis)=>best.coefficients.map((x,i)=>x+(i===axis?Math.max(1,Math.abs(x)*.2):0)))];
  for(let iteration=0;iteration<1000;iteration++) {
    simplex.sort((x,y)=>score(x)-score(y));
    const center=simplex[0].map((_,axis)=>simplex.slice(0,4).reduce((sum,c)=>sum+c[axis],0)/4);
    const point=f=>center.map((x,axis)=>x+f*(center[axis]-simplex[4][axis]));
    const reflected=point(1),value=score(reflected);
    if(value<score(simplex[0])){const expanded=point(2);simplex[4]=score(expanded)<value?expanded:reflected;}
    else if(value<score(simplex[3]))simplex[4]=reflected;
    else {
      const contracted=value<score(simplex[4])?point(.5):point(-.5);
      if(score(contracted)<Math.min(value,score(simplex[4])))simplex[4]=contracted;
      else simplex=simplex.map((c,i)=>i?c.map((x,axis)=>(x+simplex[0][axis])/2):c);
    }
  }
  simplex.sort((x,y)=>score(x)-score(y));
  if(score(simplex[0])<best.joint)best={...witness(simplex[0]),coefficients:simplex[0]};
  if(best.joint < -1e-8)return {supported:false,phaseUpper:.5,reason:'Observed intervals are inconsistent with the assumed B92 qubit model.'};
  const joint=Math.max(0,best.joint),phaseUpper=conclusive[0]>0?Math.min(.5,joint/conclusive[0]):.5;
  return {supported:true,phaseUpper,phaseJointUpper:joint,conclusiveInterval:conclusive,lossInterval:loss,errorJointUpper:errorUpper,witness:{coefficients:best.coefficients,identity:best.identity},confidenceFailure:eps,model:'IID single-photon B92 operator witness'};
}

// Exact binomial upper tail, summed outward from its mode to avoid factorial overflow.
export function detectionProbability(n,p,threshold=0) {
  if(!Number.isSafeInteger(n)||n<0||!Number.isFinite(p)||p<0||p>1||!Number.isFinite(threshold)||threshold<0||threshold>1) throw new RangeError('Invalid binomial parameters');
  const k=Math.floor(n*threshold);
  if(n===0||k>=n||p===0)return 0;if(p===1)return 1;
  if(k===0)return -Math.expm1(n*Math.log1p(-p));
  const mode=Math.floor((n+1)*p);
  if(k>=mode) {
    const first=k+1,terms=Math.min(first,n-first);
    let logChoose=0;
    for(let i=1;i<=terms;i++)logChoose+=Math.log(n-terms+i)-Math.log(i);
    const logMass=logChoose+first*Math.log(p)+(n-first)*Math.log1p(-p);
    let relative=1,sum=1;
    for(let i=first;i<n;i++){relative*=(n-i)/(i+1)*p/(1-p);sum+=relative;if(relative<sum*1e-16)break;}
    return Math.min(1,Math.exp(logMass)*sum);
  }
  let total=1,tail=mode>k?1:0,weight=1;
  for(let i=mode;i>0;i--){weight*=i/(n-i+1)*(1-p)/p;total+=weight;if(i-1>k)tail+=weight;if(weight<total*1e-16)break;}
  weight=1;
  for(let i=mode;i<n;i++){weight*=(n-i)/(i+1)*p/(1-p);total+=weight;if(i+1>k)tail+=weight;if(weight<total*1e-16)break;}
  return Math.min(1,Math.max(0,tail/total));
}
export function expectedStats(input={}) {
  const c=normalize(input),t=10**(-c.lossDb/10),p=c.attack==='ir'?c.interception:0,e=c.noise;
  const conclusive=c.protocol==='BB84'?.5:.25+p/8+e*(.5-p/4);
  const qber=c.protocol==='BB84'?e+p/4-e*p/2:(p/8+e/2-e*p/4)/conclusive;
  const signal=c.source==='ideal'?t*c.efficiency:1-Math.exp(-c.mu*t*c.efficiency);
  const honestGain=signal+(1-signal)*c.darkCount;
  // PNS keeps one photon; surviving photons reach Bob over Eve's lossless link.
  let capacity=0,prob=Math.exp(-c.mu);
  for(let n=1;n<100;n++){prob*=c.mu/n;if(n>=2)capacity+=prob*(1-(1-c.efficiency)**(n-1));}
  const forward=c.attack==='pns'?(capacity?Math.min(1,signal/capacity):0):1;
  const actualSignal=c.attack==='pns'?capacity*forward:signal;
  const gain=actualSignal+(1-actualSignal)*c.darkCount;
  const darkShare=gain?(1-actualSignal)*c.darkCount/gain:0;
  const observedConclusive=(1-darkShare)*conclusive+darkShare*.5;
  const observedQber=observedConclusive?((1-darkShare)*conclusive*qber+darkShare*.25)/observedConclusive:0;
  return {qber:observedQber,conclusive:observedConclusive,gain,honestGain,signal,capacity,forward,multiPhotonProbability:1-Math.exp(-c.mu)*(1+c.mu)};
}
export function compareProtocols(input={}) {
  const results = ['BB84', 'B92'].map(protocol => simulate({...input, protocol}));
  const eligible = results.filter(r=>['ACCEPT','ALERT'].includes(r.status)&&r.finalBits>0&&r.finalBits>=r.config.requiredBits&&!r.gainMismatch);
  eligible.sort((a,b)=>b.finalBits-a.finalBits);
  const winner=eligible[0],tied=winner&&eligible[1]?.finalBits===winner.finalBits;
  return {
    mode: 'comparison', results,
    recommendation: winner&&!tied?winner.config.protocol:null,
    reason: tied?'Both protocols meet the modeled target with equal estimated output; no yield winner.' : winner
      ? `${winner.config.protocol} has the larger eligible estimated key budget (${winner.finalBits} bits). This compares modeled yield, not proof that its key is more secure. Both are instructional estimates; review each bound and its assumptions.`
      : 'Neither protocol passes the modeled key-budget and channel checks. Increase the signal count, reduce noise/loss, or review source assumptions; no key is recommended.'
  };
}
export function interceptionCurve(input={},steps=20) {
  if(!Number.isInteger(steps)||steps<1||steps>100)throw new RangeError('Curve steps must be 1..100');
  return ['BB84','B92'].map(protocol=>({protocol,points:Array.from({length:steps+1},(_,i)=>{
    const fraction=i/steps;
    return {fraction,qber:expectedStats({...input,protocol,attack:'ir',interception:fraction}).qber};
  })}));
}
export function simulate(input={}) {
  const c=normalize(input),expected=expectedStats(c);
  const source=rng(c.seed,0x243f6a88),channel=rng(c.seed,0x85a308d3),bob=rng(c.seed,0x13198a2e),eve=rng(c.seed,0x03707344),sampling=rng(c.seed,0xa4093822);
  const t=10**(-c.lossDb/10),bits=[],trace=[],audit=[],counts={pulses:c.pulses,emitted:0,vacuum:0,multiPhoton:0,detected:0,sifted:0,sample:0,keyBits:0,errors:0,siftedErrors:0};
  for(let i=0;i<c.pulses;i++) {
    let photons=1;
    if(c.source==='wcp'){photons=-1;let product=1;do{photons++;product*=source();}while(product>Math.exp(-c.mu));}
    if(photons)counts.emitted++;else counts.vacuum++;
    if(photons>1)counts.multiPhoton++;
    const aliceDraw=source(),aliceBit=aliceDraw<.5?0:1,basisDraw=source(),aliceBasis=c.protocol==='BB84'?(basisDraw<.5?0:1):aliceBit;
    const signalDraw=channel(),darkDraw=channel(),noiseDraw=channel();
    const bobBasisDraw=bob(),bobBasis=bobBasisDraw<.5?0:1,bobDraw=bob();
    const eveForward=eve(),eveIntercept=eve(),eveBasisDraw=eve(),eveBasis=eveBasisDraw<.5?0:1,eveDraw=eve();
    let stateBasis=aliceBasis,stateBit=c.protocol==='BB84'?aliceBit:0;
    let signalProbability=1-(1-t*c.efficiency)**photons;
    if(c.attack==='pns') signalProbability=photons>=2&&eveForward<expected.forward?1-(1-c.efficiency)**(photons-1):0;
    const signal=signalDraw<signalProbability,dark=darkDraw<c.darkCount;
    // Keep a bounded audit from the same trials; recording makes no extra RNG calls.
    const event=i<128?{pulse:i+1,photons,aliceBit,aliceBasis,bobBasis,bobOutcome:null,bobBit:null,signal,darkOnly:!signal&&dark,detected:signal||dark,conclusive:false,error:false,sampled:false,status:'lost',signalProbability,noiseFlipped:signal&&noiseDraw<c.noise,draws:{aliceBit:aliceDraw,aliceBasis:basisDraw,bobBasis:bobBasisDraw,bobOutcome:bobDraw,signal:signalDraw,dark:darkDraw,noise:noiseDraw,eveInterception:eveIntercept,eveBasis:eveBasisDraw,eveOutcome:eveDraw,eveForward},eveAction:c.attack==='pns'?(photons>=2&&eveForward<expected.forward?'split':'block'):c.attack==='ir'&&eveIntercept<c.interception&&signal?'intercept':'none'}:null;
    if(event){audit.push(event);if(i<32)trace.push(event);}
    if(!signal&&!dark)continue;
    counts.detected++;
    if(c.attack==='ir'&&eveIntercept<c.interception){stateBit=eveBasis===stateBasis?stateBit:(eveDraw<.5?0:1);stateBasis=eveBasis;}
    let measured=signal?(bobBasis===stateBasis?stateBit:(bobDraw<.5?0:1)):(bobDraw<.5?0:1);
    if(signal&&noiseDraw<c.noise)measured^=1;
    const conclusive=c.protocol==='BB84'?bobBasis===aliceBasis:measured===1;
    if(event)Object.assign(event,{bobOutcome:measured,conclusive,status:conclusive?'kept':'discarded'});
    if(!conclusive)continue;
    const inferred=c.protocol==='BB84'?measured:1-bobBasis;
    if(event)Object.assign(event,{bobBit:inferred,siftedIndex:bits.length,error:inferred!==aliceBit,status:inferred!==aliceBit?'error':'kept'});
    const wrong=Number(inferred!==aliceBit);bits.push(wrong);counts.siftedErrors+=wrong;
  }
  counts.sifted=bits.length;counts.sample=Math.floor(bits.length*c.sampleFraction);
  // Track only audited positions through the actual sample shuffle, not another sample.
  const tracked=new Map(audit.filter(e=>e.conclusive).map(e=>[e.siftedIndex,e]));
  for(let i=0;i<counts.sample;i++){
    const draw=sampling(),j=i+Math.floor(draw*(bits.length-i));
    const disclosed=tracked.get(j),moved=tracked.get(i);
    if(disclosed)Object.assign(disclosed,{sampled:true,samplingDraw:draw,sampleOrder:i+1});
    tracked.delete(i);tracked.delete(j);if(moved&&i!==j)tracked.set(j,moved);
    [bits[i],bits[j]]=[bits[j],bits[i]];counts.errors+=bits[i];
  }
  counts.keyBits=counts.sifted-counts.sample;
  const qber=counts.sample?counts.errors/counts.sample:0,qUpper=qberUpper(qber,counts.sample,c.eps),gain=counts.detected/c.pulses;
  const delta=c.source==='wcp'?(gain?Math.max(0,1-expected.multiPhotonProbability/gain):0):1;
  const gainMismatch=Math.abs(gain-expected.honestGain)>Math.sqrt(Math.log(2/c.eps)/(2*c.pulses));
  const finitePenalty=2*Math.log2(1/c.eps),verification=34,authentication=64;
  const privateFraction=delta>0&&qUpper/delta<.5?delta*(1-h2(qUpper/delta)):0;
  const b92Bound=c.protocol==='B92'?b92PhaseBound({source:c.source,pulses:c.pulses,detected:counts.detected,sifted:counts.sifted,errors:counts.errors,sample:counts.sample,eps:c.eps}):null;
  const privacy=c.protocol==='B92'?(b92Bound.supported?1-h2(b92Bound.phaseUpper):0):privateFraction;
  const rawBudget=counts.keyBits*(privacy-1.16*h2(qber))-finitePenalty-verification-authentication;
  const budget={privacyFraction:privacy,privacyBits:counts.keyBits*privacy,reconciliationLeakage:counts.keyBits*1.16*h2(qber),rawBudget,beforeChecks:counts.sample?Math.max(0,Math.floor(rawBudget)):0};
  let finalBits=budget.beforeChecks;
  let status='ACCEPT';const warnings=['Educational simulation only; no cryptographic keys or security certification.'];
  if(c.protocol==='BB84'&&counts.sample&&qber>=.05)status='ALERT';
  if(!counts.sample||!counts.keyBits||finalBits<c.requiredBits)status='INSUFFICIENT';
  if(c.protocol==='BB84'&&counts.sample&&qber>.11){status='REJECT';finalBits=0;}
  if(c.protocol==='B92') {
    if(!b92Bound.supported){status=counts.sample?'UNSUPPORTED':'INSUFFICIENT';finalBits=0;warnings.push(b92Bound.reason);}
    if(counts.sample&&qber>.15){status='REJECT';finalBits=0;}
    warnings.push('B92 uses a single-photon IID phase-error witness with sampling intervals. Its finite-bit budget is instructional, not a composable proof against arbitrary coherent attacks.');
    warnings.push('B92 detector losses are modeled as erasure; dark-only clicks as random qubit replacement. An illustrative 15% error alarm is additional to the key-budget check.');
  }
  if(c.attack==='pns')warnings.push('PNS blocks single photons and forwards multiphoton pulses; channel loss is replaced by a lossless Eve-to-Bob link.');
  if(gainMismatch)warnings.push('Observed detection gain differs from the honest channel expectation.');
  if(c.source==='wcp')warnings.push('No decoy states: all multiphoton emissions are conservatively tagged as compromised.');
  if(c.darkCount)warnings.push('Dark-only detections produce random outcomes; simultaneous signal and dark clicks use the signal outcome.');
  return {config:c,counts,trace,audit,budget,detected:counts.detected,sifted:counts.sifted,sample:counts.sample,keyBits:counts.keyBits,errors:counts.errors,qber,siftedQber:counts.sifted?counts.siftedErrors/counts.sifted:0,qUpper,b92Bound,expectedQber:expected.qber,expectedConclusive:expected.conclusive,gain,expectedGain:expected.honestGain,attackExpectedGain:expected.gain,gainMismatch,delta,finalBits,status,warnings,finitePenalty,verification,authentication,pipeline:{sent:c.pulses,detected:counts.detected,sifted:counts.sifted,sampled:counts.sample,remaining:counts.keyBits,final:finalBits},detectionProbability:detectionProbability(counts.sample,expected.qber,c.protocol==='BB84'?.11:.15)};
}
"""
EXPECTED_ENGINE_SHA256 = 'd8dd2b8e7c46fe1765551b87cf7fde1e3faf5b66692eddce589bc28200f3c61a'
actual_hash = hashlib.sha256(ENGINE_SOURCE.encode("utf-8")).hexdigest()
assert actual_hash == EXPECTED_ENGINE_SHA256, "The embedded engine was modified."
pathlib.Path("qkd_colab_engine.mjs").write_text(ENGINE_SOURCE, encoding="utf-8")
pathlib.Path("qkd_colab_runner.mjs").write_text("""import {simulate} from './qkd_colab_engine.mjs';
let input='';
for await (const chunk of process.stdin) input+=chunk;
console.log(JSON.stringify(simulate(JSON.parse(input))));
""", encoding="utf-8")

def run_qkd(settings):
    completed = subprocess.run([node, "qkd_colab_runner.mjs"], input=json.dumps(settings),
                               text=True, capture_output=True, timeout=120)
    if completed.returncode:
        raise RuntimeError(completed.stderr.strip())
    return json.loads(completed.stdout)

print("Engine loaded:", version)
print("Engine SHA-256:", actual_hash)
print("Results will be calculated by this runtime, not read from the website.")


Engine loaded: v24.22.1
Engine SHA-256: d8dd2b8e7c46fe1765551b87cf7fde1e3faf5b66692eddce589bc28200f3c61a
Results will be calculated by this runtime, not read from the website.


## 2. Choose inputs and run the simulation
Change the form below, then press the **Run button on THIS cell**. It applies the inputs, runs the simulation, prints fresh bit sequences and creates fresh matching-dashboard links together. Editing a field alone does not execute code.

Start with BB84 defaults. For healthy B92 use 100,000 pulses, noise 0.005, ideal source and no attack. The output includes **INPUTS USED FOR THIS RESULT** so you can verify exactly what was applied. The first 32 pulses are a preview; the full run uses all selected pulses.


In [2]:
#@title Change inputs and run simulation
protocol = "BB84" #@param ["BB84", "B92"]
pulses = 10000 #@param {type:"integer"}
noise = 0.05 #@param {type:"number"}
attack = "none" #@param ["none", "ir", "pns"]
interception = 1.0 #@param {type:"number"}
seed = 42 #@param {type:"integer"}
required_bits = 128 #@param {type:"integer"}
source = "ideal" #@param ["ideal", "wcp"]
loss_db = 0.0 #@param {type:"number"}
efficiency = 1.0 #@param {type:"number"}
dark_count = 0.0 #@param {type:"number"}
mu = 0.5 #@param {type:"number"}
sample_fraction = 0.25 #@param {type:"number"}
eps = 0.000001 #@param {type:"number"}

# none = no attack; ir = intercept-resend (fraction above); pns = photon splitting.
# PNS requires source="wcp"; the B92 weak-coherent key bound is unsupported.
config = dict(protocol=protocol, pulses=pulses, noise=noise, attack=attack,
              interception=interception, seed=seed, requiredBits=required_bits,
              source=source, lossDb=loss_db, efficiency=efficiency,
              darkCount=dark_count, mu=mu, sampleFraction=sample_fraction, eps=eps)
# Keep the demonstration inputs within the application's supported controls.
assert 100 <= pulses <= 1000000 and 0 <= noise <= 0.5
assert 0 <= seed <= 4294967295 and 1 <= required_bits <= 1000000
assert 0 <= loss_db <= 60 and 0.01 <= mu <= 5
assert 0.01 <= sample_fraction <= 0.9 and 1e-12 <= eps <= 0.1


result = run_qkd(config)
c, n, b = result["config"], result["counts"], result["budget"]
print(f"{c['protocol']} | seed {c['seed']} | {n['pulses']:,} pulse trials")
print(f"Detected: {n['detected']:,} | Retained/sifted: {n['sifted']:,}")
qber_text = f"{100*result['qber']:.2f}%" if n['sample'] else "unavailable (no tested bits)"
print(f"QBER: {n['errors']} errors / {n['sample']} tested bits = {qber_text}")
print(f"Remaining raw bits: {n['sifted']} - {n['sample']} = {n['keyBits']}")
print(f"Privacy allowance: {n['keyBits']} x {b['privacyFraction']:.6f} = {b['privacyBits']:.3f}")
print(f"Deductions: reconciliation {b['reconciliationLeakage']:.3f}; finite-size {result['finitePenalty']:.3f}; verification {result['verification']}; authentication {result['authentication']}")
print(f"Raw budget: {b['rawBudget']:.3f}; rounded/clamped before checks: {b['beforeChecks']}")
print(f"FINAL ESTIMATE: {result['finalBits']:,} bits | {result['status']} | target {c['requiredBits']} bits")
print("\nACTUAL PULSE PREVIEW (first 32; this is not a 32-bit final key)")
print(f"{'Pulse':>5} {'Alice':>5} {'State/basis':>12} {'Bob basis':>9} {'Measured':>8} {'Bob bit':>7}  Decision")
preview = result['audit'][:32]
for e in preview:
    state = ('X' if e['aliceBasis'] else 'Z') if c['protocol']=='BB84' else ('|+>' if e['aliceBit'] else '|0>')
    decision = 'lost' if not e['detected'] else ('inconclusive/discard' if c['protocol']=='B92' else 'basis mismatch/discard') if not e['conclusive'] else ('test/disclosed' if e['sampled'] else 'keep/raw')
    if e['error']: decision += ' (error)'
    measured = '-' if e['bobOutcome'] is None else str(e['bobOutcome'])
    inferred = '-' if e['bobBit'] is None else str(e['bobBit'])
    print(f"{e['pulse']:5} {e['aliceBit']:5} {state:>12} {('X' if e['bobBasis'] else 'Z'):>9} {measured:>8} {inferred:>7}  {decision}")
kept = [e for e in preview if e['conclusive']]
print("\nAlice prepared (preview):", ''.join(str(e['aliceBit']) for e in preview))
print("Alice sifted (preview):  ", ''.join(str(e['aliceBit']) for e in kept))
print("Bob sifted (preview):    ", ''.join(str(e['bobBit']) for e in kept))
kept_count = len(kept)
lost_count = sum(not e['detected'] for e in preview)
discarded_count = len(preview) - kept_count - lost_count
errors_count = sum(e['conclusive'] and e['error'] for e in preview)
disclosed_count = sum(e['conclusive'] and e['sampled'] for e in preview)
assert kept_count + discarded_count + lost_count == len(preview)
print("\n32-PULSE PREVIEW SUMMARY" if len(preview)==32 else "PULSE PREVIEW SUMMARY")
print(f"Total transmitted pulses: {len(preview)}")
print(f"Kept / sifted bits: {kept_count}")
print(f"Discarded pulses: {discarded_count}")
print(f"Lost / not detected: {lost_count}")
print(f"Disagreements among kept bits: {errors_count}")
print(f"Disclosed for testing: {disclosed_count}")
print(f"Remaining raw preview bits: {kept_count-disclosed_count}")
print("Kept + discarded + lost = total preview pulses.")
print("These are pulse/bit counts, NOT separate accepted or rejected keys.")
print("Full-run QBER and final-key estimates above use ALL transmitted pulses.")
print("\nRaw preview bits can disagree. Reconciliation and extraction are modeled, not executed.")

print('INPUTS USED FOR THIS RESULT:', json.dumps(result['config'], sort_keys=True))

from urllib.parse import quote
fragment = '#qkd=' + quote(json.dumps(result['config'], separators=(',', ':')), safe='')
dashboard_link = 'https://vercel-demo-navy-psi.vercel.app/' + fragment
walkthrough_link = 'https://vercel-demo-navy-psi.vercel.app/simulation.html' + fragment
try:
    from IPython.display import display, HTML
    display(HTML(f'<p><a href="{dashboard_link}" target="_blank" rel="noopener">Open matching application dashboard</a></p><p><a href="{walkthrough_link}" target="_blank" rel="noopener">Open matching pulse walkthrough</a></p>'))
except ImportError:
    print('Matching dashboard:', dashboard_link)
    print('Matching walkthrough:', walkthrough_link)
pathlib.Path('qkd-colab-result.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
print('Saved qkd-colab-result.json. The application recomputes its result from the linked inputs.')


BB84 | seed 42 | 10,000 pulse trials
Detected: 10,000 | Retained/sifted: 4,962
QBER: 65 errors / 1240 tested bits = 5.24%
Remaining raw bits: 4962 - 1240 = 3722
Privacy allowance: 3722 x 0.555522 = 2067.652
Deductions: reconciliation 1280.519; finite-size 39.863; verification 34; authentication 64
Raw budget: 649.270; rounded/clamped before checks: 649
FINAL ESTIMATE: 649 bits | ALERT | target 128 bits

ACTUAL PULSE PREVIEW (first 32; this is not a 32-bit final key)
Pulse Alice  State/basis Bob basis Measured Bob bit  Decision
    1     1            Z         Z        1       1  keep/raw
    2     0            X         X        0       0  keep/raw
    3     0            X         Z        0       -  basis mismatch/discard
    4     0            Z         Z        0       0  keep/raw
    5     1            X         X        1       1  test/disclosed
    6     0            X         X        0       0  test/disclosed
    7     0            Z         Z        0       0  test/disclosed
 

## 3. Check reproducibility and expected protocol behaviour
The first check repeats the displayed experiment exactly. The other checks compare sampled outcomes with ideal-model expectations, allowing statistical variation. These are useful checks, not a security proof.

In [3]:
assert run_qkd(result['config']) == result, "The displayed run did not reproduce."
print("PASS: same settings and seed reproduce every recorded outcome and counter.")
for p, a, expected_kept, expected_qber in [
    ('BB84', 'none', 0.5, 0.0), ('BB84', 'ir', 0.5, 0.25),
    ('B92', 'none', 0.25, 0.0), ('B92', 'ir', 0.375, 1/3),
]:
    check = run_qkd(dict(protocol=p, attack=a, noise=0, pulses=100000, seed=42,
                         source='ideal', interception=1, lossDb=0, efficiency=1, darkCount=0))
    observed_kept = check['counts']['sifted']/check['counts']['pulses']
    assert abs(observed_kept-expected_kept) < 0.015, (p, a, 'sifting mismatch')
    assert abs(check['qber']-expected_qber) < 0.015, (p, a, 'QBER mismatch')
    print(f"PASS: {p}, {a}: retained {observed_kept:.3%} (expected {expected_kept:.3%}); QBER {check['qber']:.3%} (expected {expected_qber:.3%}).")


PASS: same settings and seed reproduce every recorded outcome and counter.
PASS: BB84, none: retained 49.959% (expected 50.000%); QBER 0.000% (expected 0.000%).
PASS: BB84, ir: retained 49.959% (expected 50.000%); QBER 24.333% (expected 25.000%).
PASS: B92, none: retained 24.819% (expected 25.000%); QBER 0.000% (expected 0.000%).
PASS: B92, ir: retained 37.156% (expected 37.500%); QBER 34.223% (expected 33.333%).


## 4. Compare with the application
Use **Open matching application dashboard** beneath the output in step 2. Every rerun of that input cell regenerates the link with the settings actually used. No result values are passed to the website. The JSON file is also regenerated. If inspecting a previous tab, open the NEW link after changing settings.


## Optional: automatically compare the exported dashboard JSON
On the matching dashboard, click **Export run**. Enable the checkbox below and run this cell to select that JSON file. It checks actual counts and recorded pulse evidence, not just rounded labels. Keep this off for the simple demonstration. A mismatch can mean different inputs or a newer engine version; do not call it a match until resolved.


In [4]:
compare_exported_json = False #@param {type:"boolean"}
if compare_exported_json:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Select exactly one application JSON export.')
    app = json.loads(next(iter(uploaded.values())))
    for key in ['config', 'counts', 'audit', 'finalBits', 'status']:
        assert app.get(key) == result[key], f'Mismatch in {key}: check inputs and engine version.'
    for key in ['qber', 'qUpper', 'finitePenalty', 'verification', 'authentication']:
        assert math.isclose(app[key], result[key], rel_tol=1e-10, abs_tol=1e-10), f'Mismatch: {key}'
    for key in result['budget']:
        assert math.isclose(app['budget'][key], result['budget'][key], rel_tol=1e-10, abs_tol=1e-10), f'Budget mismatch: {key}'
    print('MATCH: inputs, all counters, recorded pulse outcomes, QBER, budget and final estimate agree.')
else:
    print('Optional file comparison is off. Use the matching dashboard link for the simple demonstration.')


Optional file comparison is off. Use the matching dashboard link for the simple demonstration.


### What to tell your guide
“This notebook executes our QKD simulation outside the website. The randomized pulse outcomes produce the retained-bit and error counts. Those measured counts feed the key-length calculation. The website repeats the calculation from the same settings and seed, so we can compare the results. This is the same implementation on two platforms, supported by reproducibility and expected-behaviour checks.”
